In [1]:
# Import required libraries
import os
import warnings
import pandas as pd
import numpy as np

# Ignore irrelevant warnings
warnings.filterwarnings('ignore')

# Import model and evaluation tools
from sklearn.model_selection import StratifiedKFold
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score
from lightgbm import LGBMClassifier

In [2]:
# Fixed exact dataset path
BASE_PATH = "/kaggle/input/competitions/spaceship-titanic"

# Load train, test and sample submission files
train = pd.read_csv(os.path.join(BASE_PATH, "train.csv"))
test = pd.read_csv(os.path.join(BASE_PATH, "test.csv"))
sample_sub = pd.read_csv(os.path.join(BASE_PATH, "sample_submission.csv"))

In [3]:
# Copy original data
df_train = train.copy()
df_test = test.copy()

In [4]:
# Define numerical and categorical columns
num_cols = ["Age", "RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck"]
cat_cols = ["HomePlanet", "CryoSleep", "Destination", "VIP"]

In [5]:
# Fill missing values
imp_num = SimpleImputer(strategy="median")
imp_cat = SimpleImputer(strategy="most_frequent")

df_train[num_cols] = imp_num.fit_transform(df_train[num_cols])
df_test[num_cols] = imp_num.transform(df_test[num_cols])

df_train[cat_cols] = imp_cat.fit_transform(df_train[cat_cols])
df_test[cat_cols] = imp_cat.transform(df_test[cat_cols])

# Split Cabin into Deck, CabinNum, Side
def split_cabin(df):
    df[["Deck", "CabinNum", "Side"]] = df["Cabin"].str.split("/", expand=True)
    return df

df_train = split_cabin(df_train)
df_test = split_cabin(df_test)

In [6]:
# 1. Create total spend feature
df_train["TotalSpend"] = df_train[num_cols].sum(axis=1)
df_test["TotalSpend"] = df_test[num_cols].sum(axis=1)

# 2. Create no spend flag
df_train["NoSpend"] = (df_train["TotalSpend"] == 0).astype(int)
df_test["NoSpend"] = (df_test["TotalSpend"] == 0).astype(int)

# 3. Extract group id from PassengerId
df_train["Group"] = df_train["PassengerId"].str.split("_").str[0]
df_test["Group"] = df_test["PassengerId"].str.split("_").str[0]

# 4. Age bin feature
def age_bin(age):
    if pd.isna(age):
        return 0
    elif age < 18:
        return 1
    elif age < 30:
        return 2
    elif age < 50:
        return 3
    else:
        return 4

df_train["AgeBin"] = df_train["Age"].apply(age_bin)
df_test["AgeBin"] = df_test["Age"].apply(age_bin)

# 5. Spend level feature
def spend_level(spend):
    if spend == 0:
        return 0
    elif spend < 500:
        return 1
    elif spend < 2000:
        return 2
    else:
        return 3

In [7]:
df_train["SpendLevel"] = df_train["TotalSpend"].apply(spend_level)
df_test["SpendLevel"] = df_test["TotalSpend"].apply(spend_level)

# Encode categorical features safely
encode_cols = cat_cols + ["Deck", "Side", "Group"]
for col in encode_cols:
    df_train[col] = df_train[col].fillna("Unknown").astype("category").cat.codes
    df_test[col] = df_test[col].fillna("Unknown").astype("category").cat.codes

# Final high-level feature set
feat_cols = num_cols + cat_cols + ["Deck", "Side", "TotalSpend", 
                                   "NoSpend", "Group", "AgeBin", "SpendLevel"]

In [8]:
# Prepare training data
X = df_train[feat_cols]
y = df_train["Transported"].astype(int)
X_test = df_test[feat_cols]

# Initialize 5-fold stratified cross validation
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
test_preds = np.zeros(len(X_test))
val_acc_list = []

In [9]:
# Tuned high-performance LGBM
model = LGBMClassifier(
    n_estimators=1500,
    learning_rate=0.012,
    max_depth=10,
    num_leaves=64,
    subsample=0.9,
    colsample_bytree=0.9,
    reg_alpha=0.3,
    reg_lambda=0.3,
    random_state=42,
    verbosity=-1
)

In [10]:
# Train with 5-fold cross validation
for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
    model.fit(X_tr, y_tr)
    y_val_pred = model.predict(X_val)
    fold_acc = accuracy_score(y_val, y_val_pred)
    val_acc_list.append(fold_acc)
    print(f"Fold {fold+1} Accuracy: {fold_acc:.4f}")
    
    test_preds += model.predict_proba(X_test)[:,1] / 5

Fold 1 Accuracy: 0.8120
Fold 2 Accuracy: 0.8074
Fold 3 Accuracy: 0.8010
Fold 4 Accuracy: 0.8153
Fold 5 Accuracy: 0.7917


In [11]:
# Print mean validation accuracy
mean_acc = np.mean(val_acc_list)
print(f"\nMean Cross Validation Accuracy: {mean_acc:.4f}")


Mean Cross Validation Accuracy: 0.8055


In [12]:
# Generate submission result
test_final = (test_preds > 0.5).astype(bool)
sample_sub["Transported"] = test_final
sample_sub.to_csv("/kaggle/working/submission.csv", index=False)
print("✅ Submission file saved successfully")

✅ Submission file saved successfully
